# 🚢 Snowflake to Model Deployment Demo - With Proper Categorical Handling

In this demo, you'll walk through a complete machine learning pipeline—from data ingestion to deployment and inference—using containerized infrastructure. **This version uses Snowflake's recommended approach for categorical features with XGBoost using OneHotEncoder.**

## 📝 Prerequisites

Before starting, please ensure the following:

- Load the `titanic_snowflake.csv` dataset into your notebook environment.

Once the data is loaded, the notebook is designed to run **top-down** without interruption.

---

## 🔹 Demo Overview

This demo includes the following key steps:

1. **Data Ingestion from Snowflake**  
   Pull structured Titanic dataset from Snowflake.

2. **Feature Engineering with OneHotEncoder**  
   Transform categorical features using sklearn's OneHotEncoder, following Snowflake's recommended pattern for XGBoost.

3. **Model Training with XGBoost**  
   Use XGBoost with properly encoded categorical features that work seamlessly with Snowflake ML Registry.

4. **Model Deployment**  
   Register and deploy the trained model with proper preprocessing pipeline.

5. **Batch Inference**  
   Call the deployed model to make predictions on new batches of data.


In [ ]:
# Not neccessary since these packages come with the runtime (Just an example)
#!pip install xgboost snowflake-ml-python 


In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from snowflake.ml.registry import Registry
import ast
#add another package
# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
titanic = pd.read_csv('titanic_snowflake.csv')
titanic = titanic.drop(["AGE", 
                        "DECK", 
                        "ALIVE",
                        "ADULT_MALE",
                        "EMBARKED",
                        "PCLASS",
                        "ALONE",
                        "SEX"],axis=1)
print(f"Initial data shape: {titanic.shape}")
print(f"Columns: {list(titanic.columns)}")
titanic.head()


In [ ]:
# This step turns pandas -> snowpark and writes to snowflake
titanic_sf = session.create_dataframe(titanic)
titanic_sf.write.mode("overwrite").save_as_table("titanic_raw")


In [ ]:
# Here we read a table from Snowflake into a Snowpark dataframe

titanic_raw = session.table('titanic_raw').to_pandas()
titanic_raw.head()


In [ ]:
titanic.dropna(inplace=True)
print(f"Data shape after dropping nulls: {titanic.shape}")


In [ ]:
# Use Snowflake's recommended approach: OneHotEncoder for categorical features
# This creates a proper preprocessing pipeline that works with Snowflake ML Registry

categorical_columns = ['CLASS', 'WHO', 'EMBARK_TOWN']
numerical_columns = ['SIBSP', 'PARCH', 'FARE']

print("Setting up OneHotEncoder for categorical features...")
print(f"Categorical columns: {categorical_columns}")
print(f"Numerical columns: {numerical_columns}")

# Show unique values for categorical columns
for col in categorical_columns:
    print(f"{col} unique values: {sorted(titanic[col].unique())}")

# Create preprocessing pipeline using OneHotEncoder
# This is the recommended approach for XGBoost in Snowflake
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_columns),  # Keep numerical columns as-is
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_columns)  # One-hot encode categorical
    ],
    remainder='drop'
)

print(f"\nOriginal data shape: {titanic.shape}")
print("Preprocessing pipeline created successfully!")

titanic.head()


In [ ]:
# Prepare features and target
# Keep only the columns we need for preprocessing
feature_columns = numerical_columns + categorical_columns
x = titanic[feature_columns]
y = titanic.SURVIVED

print(f"Features shape: {x.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {list(x.columns)}")
print(f"Numerical features: {numerical_columns}")
print(f"Categorical features: {categorical_columns}")

# Show data types before preprocessing
print(f"\nData types before preprocessing:")
print(x.dtypes)


In [ ]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,train_size=.70,random_state=1234)

print(f"Training set shape: {xtrain.shape}")
print(f"Test set shape: {xtest.shape}")


In [ ]:
# Parameter grid for hyperparameter tuning
param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.1, 0.5],
    "max_depth": [1,2,3,4,5,6],
    "min_child_weight": [1, 6]
}

print(f"Parameter grid defined with {len(param_grid)} parameters")
print(f"Total combinations: {len(param_grid['n_estimators']) * len(param_grid['learning_rate']) * len(param_grid['max_depth']) * len(param_grid['min_child_weight'])}")


In [ ]:
# Create XGBoost model (without enable_categorical since we're using OneHotEncoder)
xgb_model = XGBClassifier(
    objective='binary:logistic', 
    eval_metric='logloss'
)

# Create complete pipeline: Preprocessor + XGBoost
# This is Snowflake's recommended approach for categorical features
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

print("Created complete pipeline: OneHotEncoder + XGBoost")
print("This follows Snowflake's recommended pattern for categorical features")

# Update parameter grid to work with pipeline
# Prefix XGBoost parameters with 'classifier__'
param_grid_pipeline = {
    "classifier__n_estimators": [100, 200],
    "classifier__learning_rate": [0.1, 0.5],
    "classifier__max_depth": [1, 2, 3, 4, 5, 6],
    "classifier__min_child_weight": [1, 6]
}

grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid_pipeline,
    cv=3,  # 3-fold cross validation
    scoring='accuracy',
    n_jobs=-1  # Use all available cores
)

print("Starting hyperparameter tuning with GridSearchCV...")
print("Pipeline will handle preprocessing automatically during cross-validation")
grid_search.fit(xtrain, ytrain)
print("Model training completed!")


In [ ]:
# Best parameters and score
best_params = grid_search.best_params_
best_score = grid_search.best_score_
print("Best Parameters:", best_params)
print("Best Cross-Validation Score:", best_score)

# Evaluate the best model on the test set
best_pipeline = grid_search.best_estimator_
test_score = best_pipeline.score(xtest, ytest)
print("Test Score:", test_score)

# Display feature importance for OneHotEncoded features
print("\nFeature Importance:")

# Get feature names after OneHotEncoding
# Fit the preprocessor to get the feature names
preprocessor_fitted = best_pipeline.named_steps['preprocessor']

# Get feature names from the OneHotEncoder
numerical_features = numerical_columns
categorical_features = preprocessor_fitted.named_transformers_['cat'].get_feature_names_out(categorical_columns)

# Combine all feature names
all_feature_names = list(numerical_features) + list(categorical_features)

# Get feature importance from the XGBoost model
xgb_classifier = best_pipeline.named_steps['classifier']
feature_importance = pd.DataFrame({
    'feature': all_feature_names,
    'importance': xgb_classifier.feature_importances_
}).sort_values('importance', ascending=False)

print(f"Total features after OneHotEncoding: {len(all_feature_names)}")
print("\nTop 10 most important features:")
print(feature_importance.head(10))


In [ ]:
metrics = {
    "Accuracy": best_score,
    "Test_Accuracy": test_score,
    "Params": best_params,
    "Preprocessing": "OneHotEncoder",
    "Pipeline_Components": ["ColumnTransformer", "OneHotEncoder", "XGBClassifier"],
    "Total_Features": len(all_feature_names),
    "Original_Features": len(feature_columns)
}

print("Model Metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")

metrics


In [ ]:
from snowflake.ml.registry import Registry

# Get sample input data to pass into the registry logging function
# Use original categorical string data (the pipeline will handle preprocessing)
X = xtrain.sample(n=1)
print(f"Sample input data types: {X.dtypes}")
print(f"Sample input shape: {X.shape}")
print("Sample data (pipeline will handle OneHotEncoding automatically):")
print(X)

# Create a registry and log the model
# You can specify a different DB and Schema if you'd like
# otherwise it uses the session context
# If a registry does not exist it will create one
reg = Registry(session=session)

# Define model name and version (use uppercase for name)
model_name = "TITANIC_ONEHOT"

print(f"Registering model: {model_name}")
print("Model uses OneHotEncoder preprocessing pipeline - Snowflake's recommended approach")

titanic_model = reg.log_model(
    model_name=model_name,
    options = {
    "relax_version": True,
    },
    target_platforms=["WAREHOUSE"],  
    #version_name="V_1", # If you leave version_name off SF creates one
    model=best_pipeline,  # Register the complete pipeline
    sample_input_data=X,
    metrics=metrics,
)

print(f"Model {model_name} registered successfully!")
print("Complete preprocessing pipeline registered with Snowflake ML Registry")


In [ ]:
models_df = reg.show_models()
models_df[models_df['name'] == model_name]


In [ ]:
models = reg.get_model(model_name).show_versions()
models.sort_values(by='created_on', ascending=False)


In [ ]:
recent_model = reg.get_model(model_name).last()
recent_model


In [ ]:
# Demonstrate the OneHotEncoder pipeline approach
print("=== DEMONSTRATING ONEHOT ENCODER PIPELINE ===")
print("\n1. Original test data (with string categorical values):")
print("Data types:", xtest.dtypes)
print("Sample data:")
print(xtest.head(3))

print("\n2. Pipeline automatically handles preprocessing:")
print("- Numerical columns pass through unchanged")
print("- Categorical columns are one-hot encoded")
print("- No manual dtype conversion needed!")

print("\n3. Pipeline predictions work seamlessly with string categorical data")
print("="*70)


In [ ]:
m = reg.get_model(model_name).last()
m.default = m
mv = m.default
print(f"Promoted model version: {mv.version_name}")
mv.version_name


In [ ]:
# Make predictions using the model with categorical features
print("Making remote predictions with categorical-enabled model...")

# IMPORTANT: Convert string columns back to categorical before prediction
# When data goes through Snowflake, categorical dtypes become strings
# We need to restore the categorical dtypes with the same categories as training data

# Get the original categorical mappings from training data
categorical_mappings_for_prediction = {}
for col in categorical_columns:
    categorical_mappings_for_prediction[col] = xtrain[col].cat.categories.tolist()

print(f"Categorical mappings: {categorical_mappings_for_prediction}")

# Use helper function to ensure proper categorical dtypes
xtest_categorical = ensure_categorical_dtypes(xtest, categorical_mappings_for_prediction)

print(f"Test data dtypes after conversion: {xtest_categorical.dtypes}")

# Now make predictions with properly typed categorical data
remote_prediction = mv.run(xtest_categorical, function_name="PREDICT_PROBA")
print(f"Prediction shape: {remote_prediction.shape}")
remote_prediction.head()


In [ ]:
# Write test data to Snowflake, preserving categorical information
print("Writing test data to Snowflake...")
print(f"Test data dtypes: {xtest.dtypes}")

test_sf = session.create_dataframe(xtest)
test_sf.write.mode("overwrite").save_as_table("test_pd_categorical")
session.table('test_pd_categorical').show()


In [ ]:
titanic_sf = session.create_dataframe(xtest)
titanic_sf.write.mode("overwrite").save_as_table("titanic_predict_categorical")


In [ ]:
select *, round(TITANIC_CATEGORICAL!predict_proba(*):output_feature_0,2)
as surv_pred
from titanic_predict_categorical
limit 10


In [ ]:
current_wh = session.get_current_warehouse()
current_wh


In [ ]:
create or replace dynamic table titanic_batch_inference_categorical
target_lag = '1 minute' 
warehouse = {{current_wh}} as
select *, round(TITANIC_CATEGORICAL!predict_proba(*):output_feature_0,2)
as surv_pred
from test_pd_categorical;

select * from titanic_batch_inference_categorical limit 5;


In [ ]:
-- Insert new data with categorical values (not dummy encoded)
INSERT INTO test_pd_categorical (
    SIBSP, PARCH, FARE, CLASS, WHO, EMBARK_TOWN
) VALUES
(0, 0, 10.5, 'Third', 'man', 'Southampton'),
(2, 1, 23.0, 'Second', 'woman', 'Southampton'),
(0, 2, 15.75, 'Second', 'woman', 'Queenstown'),
(1, 1, 7.925, 'Third', 'man', 'Southampton'),
(0, 0, 7.75, 'Third', 'man', 'Southampton'),
(3, 2, 21.6792, 'Second', 'man', 'Southampton'),
(0, 0, 8.05, 'Third', 'man', 'Queenstown'),
(0, 0, 8.6625, 'Third', 'woman', 'Queenstown'),
(1, 0, 26.0, 'Second', 'man', 'Southampton'),
(0, 1, 19.2583, 'Second', 'woman', 'Southampton');


In [ ]:
drop dynamic table if exists titanic_batch_inference_categorical;
